# BERT-Base uncased — DIMER E2E continued-pre-training tutorial: masked-token prediction on paper abstracts (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bert-masked-lm-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bert-masked-lm-pipeline/blob/main/tutorials/bert_masked_lm_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--bert%2Fbert--base--uncased-ffcc4d?style=flat)](https://huggingface.co/google-bert/bert-base-uncased) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fbert-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/bert) [![arXiv](https://img.shields.io/badge/arXiv-1810.04805-b31b1b.svg)](https://arxiv.org/abs/1810.04805)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** masked-language modelling (fill-mask: ranked candidates for one `[MASK]`), sentence embeddings (768-d, CLS or mean pooled, L2-normalised) and bounded continued masked-language-model training of the last encoder layers on a text corpus, measured by held-out masked-token perplexity and top-k accuracy, using the pinned BERT-Base uncased weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/bert_masked_lm_pipeline/`, at revision `18ee9f3e7270`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `86b5e0934494bd15c9632b12f734a8a67f723594` (~441 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (the kernel's own packages are left alone, so no restart is needed), stages and digest-verifies the pinned BERT-Base uncased snapshot (safetensors, 440 MB), fetches the three digest-pinned SciTLDR-A files from the project repository (5.5 MB, no credential), reads the paper abstracts and draws 300 / 50 / 100 training, validation and test documents from the release's own paper-disjoint members, fills one mask in an unseen abstract's opening sentence and embeds two test abstracts through the inference contracts with an input manifest and a rejection probe, scores the frozen model on the test abstracts by masked-token perplexity and top-1 / top-5 accuracy at seeded masks beside the add-one unigram floor, runs a bounded continued masked-language-model training of the last four encoder layers with validation-perplexity epoch selection, scores the held-out split again, fills the same masks and re-embeds the same abstracts with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). A CUDA runtime is used automatically when present. Measured times come from runs of the previous notebook version, whose model stages are the same: the Kaggle Tesla T4 run of 2026-09-19 took 272.5 s for all cells including the 447 MB checkpoint download, and a local Windows CPU pre-flight took 233.1 s with the files pre-staged (207 s of it the two-epoch adaptation). No hosted CPU run is recorded, so on a hosted CPU runtime expect roughly the CPU pre-flight figure (an estimate). Building the isolated environment and downloading the checkpoint come on top of these.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4, select that cell and choose **Runtime → Run after** (it re-runs Section 4 and every later cell; Sections 5, 6 and 7 reload the pretrained model first, so the earlier adaptation never carries over). Supply your own text corpus through the upload dialog on Colab or a file path in `BYOD_PATH` on any runtime, as a CSV (columns `id`, `text`), a JSON array or JSONL file of `{id, text}` records, or a plain `.txt` file in which every blank-line-separated paragraph is one document. It needs at least **12 distinct documents** (so the split leaves 8 training, 2 validation and 2 test documents), each at most 512 WordPiece tokens; Section 4 checks both, names the split or record it refuses, and reports how many duplicate texts it dropped, before any model runs. It passes through the same validation, seeded text-disjoint split, unigram floor, frozen scoring, fine-tuning, held-out evaluation, cloze and embedding checks, artifact export and reload-parity cells as the SciTLDR sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the WordPiece tokenizer lower-cases the text and one forward pass of the 12-layer bidirectional encoder runs; `fill_mask` reads the output-vocabulary logits at the single `[MASK]` position through the masked-LM head and ranks them by a softmax over the 30,522-token vocabulary, while `embed` takes the encoder's last hidden states and pools them to one 768-d vector per text (the `[CLS]` position, or the attention-masked mean) and L2-normalises it. The carried pipeline module adds manifest verification, input validation and ceilings (over-long texts are rejected, not truncated), the two task methods and fixed output contracts. **The fill-mask `score` is not a calibrated probability** (it is a softmax ranking signal), and **embeddings are representations, not predictions**; the pipeline ships no threshold for either.

What this notebook adds to inference is **continued pre-training measured by the model's own objective**. The dataset is real: SciTLDR-A (Cachola et al., 2020; Apache-2.0) ships the abstracts of 3,229 computer-science papers as three digest-pinned JSON-Lines files fetched from the project repository at a pinned commit; only the abstract text is used, so every record is one document and no label is needed. The carried `metrics.py` masks a **seeded 15 %** of each held-out abstract's WordPiece tokens — the same positions for every model scored under the same seed — and reads the negative log-likelihood and the rank of the original token at every masked position, aggregated into **masked perplexity**, **bits per masked token** and **top-1 / top-5 accuracy**; an **add-one unigram model** fitted on the training tokens is the floor a model that ignores context reaches. The fine-tuning question is whether a bounded masked-language-model adaptation of the last encoder layers on 300 abstracts lowers that perplexity on abstracts the model has not seen. Nothing here is a quality claim: a lower masked perplexity says the adapted encoder predicts missing words of paper abstracts better, not that its embeddings are better for your task.

**Who this is for.** A learner who knows basic Python and has met the idea of a language model, and wants to see how a pretrained encoder is measured and adapted to a new kind of text. No prior experience with BERT, perplexity or fine-tuning is assumed; each term is explained where it is first used and again in the **Glossary** at the end.

**Input → Model → Output.**

| | Fill-mask | Embedding | Continued pre-training |
|---|---|---|---|
| Input | one text with exactly one `[MASK]` | 1..64 texts | 300 training abstracts (no labels) |
| Model | BERT-Base uncased encoder + masked-LM head | the same encoder, pooled | the last 4 encoder layers trained on masked words |
| Output | the top-`k` candidate tokens with softmax scores | one unit-length 768-d vector per text | an adapter (safetensors) and held-out masked perplexity / top-k accuracy |

**How to use this notebook.** Choose a runtime (a GPU runtime is faster; CPU works), then **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried code and the model verification — and can be run without study. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the recorded path. Each stage states what it does, asks you to **predict** before it runs, and the next cell opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer. Each optional experiment names the cell to change and the cells to re-run. Section 10 is a **change-one-thing activity**. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional and are not required submissions.

**Roadmap:** 4 fetch, validate and split the corpus → 5 fill a mask and embed with the pretrained model → 6 the unigram floor and the pretrained model's masked perplexity → 7 continue pre-training the last four layers → 8 compare on held-out abstracts → 9 clozes and embeddings before and after, export and reload → 10 **change one thing: fewer trainable layers** → conclude.

**Learning objectives:** by the end you should be able to (1) explain *text with one `[MASK]` → encoder and masked-LM head → ranked candidate tokens* and *texts → encoder → unit vectors*, and say why the fill-mask score is not a probability of being right (Section 5); (2) explain, from Section 6's numbers, why the unigram floor's masked perplexity is in the thousands while the pretrained model's is near 15; (3) predict and then check how much a bounded continued pre-training of the last encoder layers lowers masked perplexity on held-out abstracts (Sections 7–8); (4) explain why validation, never test, chooses the kept epoch; (5) predict and then measure how the gain changes with one trainable layer instead of four (Section 10); (6) state what a lower masked perplexity does and does not tell you about the embeddings and clozes of Section 9; and (7) check that an exported adapter reproduces the evaluated model (reload parity, Section 9).

**This notebook does not demonstrate:** classification or other supervised heads, next-sentence prediction, multi-mask filling, raw-logit access, text generation (BERT is an encoder), cased or non-English text (the checkpoint is uncased English), contrastively trained sentence similarity (the `qwen3-embedding-pipeline` sibling covers retrieval-grade embeddings), full-model or embedding-table training, any similarity or downstream-task score, and any claim that a SciTLDR abstract split stands in for your corpus. The repository exposes none of these.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; no prior experience with BERT or fine-tuning. The notebook explains masked-language modelling, the softmax over a vocabulary (and why it is not a calibrated probability), perplexity (the exponential of the mean negative log-likelihood, an intrinsic number, not a judgement of embedding quality) and cosine similarity between unit vectors where they are first used; the Glossary repeats them.
- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab, Kaggle or Linux Jupyter. Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter, and a Windows or macOS kernel is not supported (Section 1 stops with that message). The default path runs on CPU (float32) and uses CUDA automatically when available. The locked install (PyTorch 2.14.0 with its CUDA libraries) and the 440 MB checkpoint are the large downloads. Measured on the previous notebook version, whose model stages are the same, in a local Windows CPU pre-flight (2026-09-19, float32): 6.6 s to score the 100-abstract test split (2,983 masked positions) and 206.9 s for the two-epoch adaptation over 300 abstracts with a 50-abstract validation pass per epoch (about 100 s per epoch); the whole Kaggle Tesla T4 run took 272.5 s including the checkpoint download. No hosted CPU run is recorded; expect roughly the CPU pre-flight figures there (an estimate).
- **Data contract:** records are `{id, text}` — one document of the domain, 1..4,000 characters and at most 512 WordPiece tokens including `[CLS]`/`[SEP]` (a longer record is refused, not truncated, everywhere), ids matching `[A-Za-z0-9_.:-]{1,64}` and unique; a dataset holds at most 20,000 records; texts are de-duplicated case-insensitively before splitting so the same document never sits in two splits; text is lower-cased by the tokenizer, so case carries no information. BYOD accepts CSV, JSON, JSONL or TXT in that shape and needs at least **12 distinct documents**: the seeded split keeps about 20 % for test and 15 % for validation, and must leave at least 8 training, 2 validation and 2 test documents. Section 4 checks this and the 512-token ceiling before any model runs, and names the split or the record (by split and id) it refuses.
- **Validation is structural, not semantic:** nothing checks that a record belongs to the domain you mean — an off-topic corpus is trained on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document collection is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches three pinned objects (`train.jsonl` 3,155,015 bytes, `dev.jsonl` 1,124,865 bytes, `test.jsonl` 1,204,107 bytes; SHA-256 `b222771d…` / `3191fa98…` / `fb42dd6c…`) from `raw.githubusercontent.com` at the pinned `allenai/scitldr` commit over HTTPS, each refused on any mismatch before it is read; SciTLDR is Apache-2.0 (Cachola et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `google-bert/bert-base-uncased` snapshot (~441 MB in total) at revision `86b5e0934494…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 47 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 47 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '97708ec478e01f5e76780ba32333472357c7da899529f4e6e0502f918847cd08'
LOCKED_PACKAGES = 47
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via torchvision
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via
    #   bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
    #   transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via bart-mnli-zero-shot-classification-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 47, 'setup_seconds': 54}


The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

from IPython import get_ipython as _kernel_shell

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    try:
        import google
    except ImportError:
        google = types.ModuleType("google")
        google.__path__ = []
        sys.modules["google"] = google
    _colab = types.ModuleType("google.colab")
    _files = types.ModuleType("google.colab.files")
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 1203).


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bert-masked-lm-pipeline',
    'repository_revision': '18ee9f3e72706a49af8715f53c8251c5d3b0bed8',
    'embedded_module': 'src/bert_masked_lm_pipeline/pipeline.py',
    'embedded_modules': ['src/bert_masked_lm_pipeline/metrics.py', 'src/bert_masked_lm_pipeline/pipeline.py', 'src/bert_masked_lm_pipeline/samples.py'],
    'module_sha256': 'd72d7a0e143f1313052104edd6c61903fc4a772888b40a32ae218738fce06428',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'bert-masked-lm-pipeline', 'repository_revision': '18ee9f3e72706a49af8715f53c8251c5d3b0bed8', 'embedded_module': 'src/bert_masked_lm_pipeline/pipeline.py', 'embedded_modules': ['src/bert_masked_lm_pipeline/metrics.py', 'src/bert_masked_lm_pipeline/pipeline.py', 'src/bert_masked_lm_pipeline/samples.py'], 'module_sha256': 'd72d7a0e143f1313052104edd6c61903fc4a772888b40a32ae218738fce06428', 'generator': 'build_notebook.py/2.1', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/bert_masked_lm_pipeline/` @ `18ee9f3e7270`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/bert_masked_lm_pipeline/metrics.py`

In [ ]:
"""Masked-language-model metrics over a held-out text corpus, seeded masking, and the unigram baseline.

A record is scored by masking a seeded 15 % of its WordPiece tokens (never `[CLS]`/`[SEP]`, at least one
token) and reading, at every masked position, the model's negative log-likelihood of the original token and
the original token's rank in the model's prediction. Those aggregate into **masked perplexity**
(`exp(mean NLL)` over all masked positions, token-weighted), **bits per masked token** (`mean NLL / ln 2`)
and **top-1 / top-5 accuracy** (the original token is the first / among the first five candidates). The
masks are a fixed function of the seed and the record's id, so the frozen and adapted models are scored on
exactly the same positions. The **unigram baseline** predicts every masked position with an add-one-smoothed
unigram distribution fitted on the training tokens — the floor a model that ignores context reaches.
"""

from __future__ import annotations

import hashlib
import math
import random
from collections import Counter
from collections.abc import Sequence
from typing import Any

DEFAULT_MASK_RATE = 0.15
METRIC_DEFINITIONS = {
    "perplexity": (
        "exp of the mean negative log-likelihood (natural log) of the original token at every masked "
        "position of the held-out records, all masked positions weighted equally; lower is better"
    ),
    "bits_per_token": "the same mean negative log-likelihood divided by ln 2",
    "mean_nll": "the mean negative log-likelihood at the masked positions in nats",
    "top1_accuracy": "fraction of masked positions whose original token is the model's first candidate",
    "top5_accuracy": "fraction of masked positions whose original token is among the first five candidates",
    "masking": (
        "a seeded 15 % of each record's WordPiece tokens (never [CLS]/[SEP], at least one token) replaced "
        "by [MASK] — the same positions for every model scored under the same seed"
    ),
}


def record_seed(record_id: str, seed: int) -> int:
    """A stable per-record seed so the masked positions depend only on the record and the base seed."""
    digest = hashlib.sha256(f"{seed}:{record_id}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def mask_positions(n_tokens: int, *, rate: float = DEFAULT_MASK_RATE, seed: int) -> list[int]:
    """Sorted positions to mask among 1..n_tokens-2 (the interior; position 0 is [CLS], the last is [SEP])."""
    if not 0.0 < rate <= 1.0:
        raise ValueError("rate must be in (0, 1]")
    interior = n_tokens - 2
    if interior < 1:
        raise ValueError("a record needs at least one token between [CLS] and [SEP]")
    count = max(1, round(interior * rate))
    return sorted(random.Random(seed).sample(range(1, n_tokens - 1), count))


def masked_metrics(scores: Sequence[Sequence[tuple[float, int]]]) -> dict[str, Any]:
    """Aggregate per-record lists of (NLL in nats, rank of the original token, 1 = first) pairs."""
    if not scores:
        raise ValueError("no records to score")
    total = 0.0
    n_masked = 0
    top1 = 0
    top5 = 0
    for record_scores in scores:
        if not record_scores:
            raise ValueError("a record scored no masked positions")
        for nll, rank in record_scores:
            if rank < 1:
                raise ValueError("rank must be 1 or more")
            total += float(nll)
            n_masked += 1
            top1 += rank == 1
            top5 += rank <= 5
    mean_nll = total / n_masked
    return {
        "n_records": len(scores),
        "n_masked": n_masked,
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "bits_per_token": mean_nll / math.log(2),
        "top1_accuracy": top1 / n_masked,
        "top5_accuracy": top5 / n_masked,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def unigram_masked_scores(
    train_ids: Sequence[Sequence[int]], targets: Sequence[Sequence[int]], vocab_size: int
) -> list[list[tuple[float, int]]]:
    """(NLL, rank) of each masked original token under an add-one unigram model fitted on `train_ids`.
    The rank counts vocabulary entries with a strictly higher count plus one, so ties favour the target."""
    if vocab_size < 1:
        raise ValueError("vocab_size must be positive")
    counts: Counter[int] = Counter()
    for ids in train_ids:
        counts.update(int(t) for t in ids)
    total = sum(counts.values()) + vocab_size
    log_total = math.log(total)
    ordered = sorted(counts.values(), reverse=True)
    out = []
    for record_targets in targets:
        record = []
        for target in record_targets:
            count = counts.get(int(target), 0)
            higher = sum(1 for c in ordered if c > count)
            record.append((log_total - math.log(count + 1), higher + 1))
        out.append(record)
    return out


def unigram_baseline(
    train_ids: Sequence[Sequence[int]], targets: Sequence[Sequence[int]], vocab_size: int
) -> dict[str, Any]:
    """Masked metrics of the context-free unigram model — the floor a masked language model must beat."""
    result = masked_metrics(unigram_masked_scores(train_ids, targets, vocab_size))
    result["baseline"] = "add-one-smoothed unigram model fitted on the training tokens (ignores context)"
    return result

**Module 2/3:** `src/bert_masked_lm_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Masked-language modelling and sentence embeddings over the pinned ``google-bert/bert-base-uncased``.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. Two task methods: ``fill_mask`` (one ``[MASK]`` token ->
ranked vocabulary candidates) and ``embed`` (CLS or mean pooled, L2-normalised 768-d representations).

The adaptation contract (``evaluate``, ``unigram_baseline``, ``adapt``, ``save_artifact``, ``from_artifact``)
scores a validated ``{id, text}`` corpus by masked-token prediction at seeded positions, continues the
masked-language-model objective on it for the last encoder layers with validation-perplexity epoch selection,
and exports the trained tensors as a safetensors adapter bound to the pinned base weights. The two inference
methods are unchanged by it, but both read the adapted encoder once ``adapt`` or ``load_artifact`` has run.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "google-bert/bert-base-uncased"
MODEL_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "bert-base-uncased"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. 512 is max_position_embeddings in the pinned config.json and model_max_length in
# tokenizer_config.json; longer inputs are rejected (not truncated) so a caller never silently loses [MASK].
MAX_TEXT_TOKENS = 512
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard; ~4 chars per WordPiece token on English text
MAX_BATCH = 64  # texts per embed() call
MAX_TOP_K = 100
VOCAB_SIZE = 30522  # config.json vocab_size
HIDDEN_SIZE = 768  # config.json hidden_size
MASK_TOKEN = "[MASK]"
POOLINGS = ("cls", "mean")
DEFAULT_TOP_K = 5
PAD_TOKEN_ID = 0  # vocab.txt [PAD]
CLS_TOKEN_ID = 101  # vocab.txt [CLS]
SEP_TOKEN_ID = 102  # vocab.txt [SEP]
MASK_TOKEN_ID = 103  # vocab.txt [MASK]
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 109_514_298
ENCODER_LAYERS = 12  # config.json num_hidden_layers
DEFAULT_TRAINABLE_LAYERS = 4  # the last four encoder layers (28,351,488 parameters)
MAX_EVAL_RECORDS = 2_000
MAX_RECORDS_FIT = 20_000  # the unigram baseline may be fitted on a whole training split
MIN_SCORED_RECORDS = 50  # below this a scored corpus is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.bert-base-uncased.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_text(text: Any, name: str) -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_mask_count(text: str) -> str:
    """`fill_mask` accepts exactly one [MASK]; raise naming the count found."""
    if text.count(MASK_TOKEN) != 1:
        raise ValueError(f"text must contain exactly one {MASK_TOKEN}, found {text.count(MASK_TOKEN)}")
    return text


def _check_top_k(top_k: Any) -> int:
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= MAX_TOP_K:
        raise ValueError(f"top_k must be between 1 and MAX_TOP_K={MAX_TOP_K}")
    return top_k


def _check_batch(texts: Any, pooling: Any) -> list[str]:
    """`embed`'s batch contract; raise naming the first violated ceiling."""
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..MAX_BATCH={MAX_BATCH} items, got {len(texts)}")
    clean = [_check_text(t, f"texts[{i}]") for i, t in enumerate(texts)]
    if pooling not in POOLINGS:
        raise ValueError(f"pooling must be one of {POOLINGS}")
    return clean


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "sequence of non-empty str; every entry carrying a [MASK] token is also a fill_mask input, "
        "every entry is an embed input (one vector per text)"
    ),
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "text_tokens": [1, MAX_TEXT_TOKENS],
    "top_k": [1, MAX_TOP_K],
    "pooling": list(POOLINGS),
    "mask_token": MASK_TOKEN,
    "masks_per_fill_mask_text": 1,
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": HIDDEN_SIZE,
    "preprocessing": (
        "WordPiece tokenisation that lower-cases and strips accents; texts past MAX_TEXT_TOKENS are "
        "rejected, never truncated, so a [MASK] can never be silently lost; embed pools the last "
        "layer (cls position or attention-masked mean) and L2-normalises"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    *,
    top_k: int = DEFAULT_TOP_K,
    pooling: str = "cls",
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    ``texts`` is the batch ``embed`` would take; every entry that carries a ``[MASK]`` token is
    additionally checked against ``fill_mask``'s contract (exactly one mask, ``top_k`` in range) and
    marked in the manifest. Both capabilities' checks run through the same private functions the
    methods use — ``_check_batch``/``_check_text`` for ``embed``, ``_check_mask_count``/``_check_top_k``
    for ``fill_mask`` — so a rejection here is a rejection there. ``MAX_TEXT_TOKENS`` is enforced
    after tokenisation inside the pipeline and therefore cannot be observed at this stage.
    """
    checked = _check_batch(texts, pooling)
    _check_top_k(top_k)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    inputs = []
    for i, text in enumerate(checked):
        masks = text.count(MASK_TOKEN)
        if masks:
            _check_mask_count(text)
        inputs.append(
            {
                "id": names[i] if names else f"text-{i}",
                "chars": len(text),
                "masks": masks,
                "fill_mask_input": bool(masks),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "top_k": top_k,
        "pooling": pooling,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], expected_tokens: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Neither capability has a metric helper in this repository, so the verdict is always
    ``not-measurable`` (EVAL9). ``expected_tokens`` exists for interface parity with the fleet's
    other pipelines and is recorded in ``reason`` rather than scored: one author-expected token is
    an intent, not a labelled cloze set, and computing a hit rate from it would present a single
    observation as an accuracy. ``result`` is the ``fill_mask`` result; the embedding half is a
    representation and is covered by the same verdict.
    """
    candidates = result.get("candidates", [])
    supplied = expected_tokens is not None
    return {
        "task": "masked-language modelling (fill-mask) and sentence embedding",
        "score_semantics": (
            f"fill_mask `score` is a softmax over the {VOCAB_SIZE}-token vocabulary at the masked "
            "position — a ranking signal, not a calibrated probability, with argmax as the decision "
            f"rule and no shipped threshold; embed returns {HIDDEN_SIZE}-d unit vectors whose only "
            "meaning is cosine within the same model and pooling policy"
        ),
        "sample_kind": sample_kind,
        "n_candidates": len(candidates),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the repository ships no metric helper for either capability"
            + (
                "; an expected token was supplied, but one author-expected token is an intent rather "
                "than a labelled cloze set, so scoring it would present a single observation as an accuracy"
                if supplied
                else "; the evaluated sample carries no gold tokens and no similarity labels"
            )
        ),
        "needs": (
            "for fill-mask, a labelled cloze set (sentence, mask position, gold token) over enough "
            "sentences to state a dispersion, scored with the caller's own top-1/top-k hit-rate code; "
            "for the embeddings, a judged similarity set (Spearman correlation) or a retrieval or "
            "clustering set with relevance labels (recall@k) — neither of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class BERTMaskedLMPipeline:
    """``_mask_runner`` maps one text to (vocab logits at the [MASK] position, n_tokens);
    ``_embed_runner`` maps texts to (last hidden states (N, T, 768), attention mask (N, T)); both injectable.
    ``_decode`` maps a token id to its string."""

    _mask_runner: Callable[[str], tuple[np.ndarray, int]]
    _embed_runner: Callable[[list[str]], tuple[np.ndarray, np.ndarray]]
    _decode: Callable[[int], str]
    device: str = "cpu"
    source: str = "injected"
    _encode: Callable[[str], list[int]] | None = field(default=None, repr=False)
    _mlm_scorer: Callable[[list[int], list[int], list[int]], list[tuple[float, int]]] | None = field(
        default=None, repr=False
    )
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BERTMaskedLMPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
            origin = "local-snapshot"
        elif allow_download:
            source, kwargs, origin = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, BertForMaskedLM

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BertForMaskedLM.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        mask_id = tokenizer.mask_token_id

        def mask_runner(text: str) -> tuple[np.ndarray, int]:
            batch = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            n_tokens = int(batch["input_ids"].shape[1])
            if n_tokens > MAX_TEXT_TOKENS:
                raise ValueError(f"text tokenises to {n_tokens} tokens; MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}")
            position = (batch["input_ids"][0] == mask_id).nonzero().flatten()
            with torch.inference_mode():
                logits = model(**batch).logits[0, position[0]]
            return logits.float().cpu().numpy(), n_tokens

        def embed_runner(texts: list[str]) -> tuple[np.ndarray, np.ndarray]:
            batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=False)
            if batch["input_ids"].shape[1] > MAX_TEXT_TOKENS:
                raise ValueError(f"a text tokenises past MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}")
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                hidden = model.bert(**batch).last_hidden_state
            return hidden.float().cpu().numpy(), batch["attention_mask"].cpu().numpy()

        def encode(text: str) -> list[int]:
            return [int(i) for i in tokenizer(text, truncation=False)["input_ids"]]

        def mlm_scorer(ids: list[int], positions: list[int], targets: list[int]) -> list[tuple[float, int]]:
            """(NLL of the original token, its rank) at every masked position of one already-masked record."""
            input_ids = torch.tensor([ids], dtype=torch.long, device=resolved_device)
            with torch.inference_mode():
                logits = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids)).logits[0]
            rows = logits[positions].float()
            log_probs = torch.log_softmax(rows, dim=-1)
            target = torch.tensor(targets, dtype=torch.long, device=resolved_device)
            nll = -log_probs.gather(1, target[:, None])[:, 0]
            rank = (rows > rows.gather(1, target[:, None])).sum(dim=1) + 1
            return list(zip(nll.tolist(), rank.tolist(), strict=True))

        return cls(
            mask_runner,
            embed_runner,
            tokenizer.convert_ids_to_tokens,
            resolved_device,
            origin,
            _encode=encode,
            _mlm_scorer=mlm_scorer,
            _model=model,
            _tokenizer=tokenizer,
        )

    def fill_mask(self, text: str, top_k: int = DEFAULT_TOP_K) -> dict[str, Any]:
        """Rank candidates for exactly one ``[MASK]``; ``score`` is a softmax over the 30 522-token vocab."""
        text = _check_mask_count(_check_text(text, "text"))
        top_k = _check_top_k(top_k)
        logits, n_tokens = self._mask_runner(text)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (VOCAB_SIZE,):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({VOCAB_SIZE},)")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs, kind="stable")[:top_k]
        candidates = [
            {
                "token": self._decode(int(i)),
                "token_id": int(i),
                "score": float(probs[i]),
                "sequence": text.replace(MASK_TOKEN, self._decode(int(i)), 1),
            }
            for i in order
        ]
        return {
            "candidates": candidates,
            "top_k": top_k,
            "n_tokens": n_tokens,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def embed(self, texts: Sequence[str], pooling: str = "cls") -> dict[str, Any]:
        """L2-normalised 768-d representations: CLS token or attention-masked mean of the last layer."""
        clean = _check_batch(texts, pooling)
        hidden, mask = self._embed_runner(clean)
        hidden = np.asarray(hidden, dtype=np.float32)
        mask = np.asarray(mask, dtype=np.float32)
        if hidden.ndim != 3 or hidden.shape[0] != len(clean) or hidden.shape[2] != HIDDEN_SIZE:
            raise RuntimeError(f"backend returned {hidden.shape}, expected ({len(clean)}, T, {HIDDEN_SIZE})")
        if pooling == "cls":
            pooled = hidden[:, 0]
        else:
            counts = np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
            pooled = (hidden * mask[:, :, None]).sum(axis=1) / counts
        normalized = pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)
        return {
            "embeddings": normalized.tolist(),
            "dim": HIDDEN_SIZE,
            "pooling": pooling,
            "normalized": True,
            "n_tokens": [int(v) for v in mask.sum(axis=1)],
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def count_tokens(self, text: str) -> int:
        """WordPiece tokens of ``text`` including [CLS]/[SEP], exactly as ``evaluate`` and ``adapt`` encode it
        (no truncation), without running the model, so a corpus can be checked against MAX_TEXT_TOKENS
        before any model call."""
        if self._encode is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return len(self._encode(_check_text(text, "text")))

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def _record_ids(self, records: Sequence[Mapping[str, Any]]) -> list[list[int]]:
        """Tokenise validated records with [CLS]/[SEP]; a record is refused (never truncated) above
        MAX_TEXT_TOKENS or without at least one interior token."""
        if self._encode is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        out = []
        for record in records:
            ids = list(self._encode(record["text"]))
            if len(ids) > MAX_TEXT_TOKENS:
                raise ValueError(f"record {record['id']} has {len(ids)} tokens; ceiling is {MAX_TEXT_TOKENS}")
            if len(ids) < 3:
                raise ValueError(f"record {record['id']} has no token between [CLS] and [SEP]")
            out.append(ids)
        return out

    @staticmethod
    def _masked(
        records: Sequence[Mapping[str, Any]], ids: Sequence[Sequence[int]], *, rate: float, seed: int
    ) -> list[tuple[list[int], list[int], list[int]]]:
        """(masked ids, positions, original tokens) per record under the seeded per-record masking."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import mask_positions, record_seed` removed — names are kernel globals defined by the carried modules

        out = []
        for record, record_ids in zip(records, ids, strict=True):
            positions = mask_positions(len(record_ids), rate=rate, seed=record_seed(record["id"], seed))
            masked = list(record_ids)
            for position in positions:
                masked[position] = MASK_TOKEN_ID
            out.append((masked, positions, [record_ids[p] for p in positions]))
        return out

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, mask_rate: float = 0.15, seed: int = 0
    ) -> dict[str, Any]:
        """Masked-token prediction on a validated corpus: a seeded `mask_rate` of each record's interior
        tokens is replaced by [MASK] and the original tokens are scored by NLL and rank."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import masked_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self._mlm_scorer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        scores = [
            self._mlm_scorer(masked, positions, targets)
            for masked, positions, targets in self._masked(
                checked, self._record_ids(checked), rate=mask_rate, seed=seed
            )
        ]
        metrics = masked_metrics(scores)
        metrics.update(
            {
                "mask_rate": mask_rate,
                "seed": seed,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def unigram_baseline(
        self,
        train: Sequence[Mapping[str, Any]],
        test: Sequence[Mapping[str, Any]],
        *,
        mask_rate: float = 0.15,
        seed: int = 0,
    ) -> dict[str, Any]:
        """The add-one unigram model fitted on `train` (interior tokens), scored at the same masked positions
        of `test` that `evaluate` uses under the same seed."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import unigram_baseline` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        train_checked = validate_dataset(train, min_records=1, max_records=MAX_RECORDS_FIT)["records"]
        test_checked = validate_dataset(test, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        train_ids = [ids[1:-1] for ids in self._record_ids(train_checked)]
        targets = [
            record_targets
            for _masked, _positions, record_targets in self._masked(
                test_checked, self._record_ids(test_checked), rate=mask_rate, seed=seed
            )
        ]
        result = unigram_baseline(train_ids, targets, VOCAB_SIZE)
        result.update({"mask_rate": mask_rate, "seed": seed})
        return result

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if not isinstance(trainable_layers, int) or not 1 <= trainable_layers <= ENCODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 1..{ENCODER_LAYERS}")
        model, _ = self._require_model()
        first = ENCODER_LAYERS - trainable_layers
        prefixes = tuple(f"bert.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        lr: float = 5e-5,
        batch_size: int = 8,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        mask_rate: float = 0.15,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded continued masked-language-model training on a validated text corpus.

        Only the last `trainable_layers` encoder layers train (4 by default; the word, position and
        token-type embeddings, the earlier layers, the pooler and the MLM head with its tied decoder stay
        frozen). Every epoch re-draws a seeded `mask_rate` of each record's interior tokens, replaces them
        with [MASK] and applies cross-entropy at those positions only; AdamW at a fixed learning rate,
        gradient clipping at 1.0, no scheduler; records over MAX_TEXT_TOKENS are refused, never truncated.
        Epoch 0 records the frozen model's validation metrics under the evaluation masking (`seed`); the
        epoch with the lowest validation masked perplexity is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        if not (0.0 < mask_rate <= 0.5):
            raise ValueError("mask_rate must be in (0, 0.5]")
        if self.adapter is not None:
            # adapt() trains from the weights in memory and records epoch 0 as the frozen model; on an adapted
            # (or artifact-loaded) pipeline that would stack a second adaptation under a "frozen model" label
            # and export only the latest layers (review MLM-M2).
            raise ValueError(
                "this pipeline is already adapted; adapt() starts from the pretrained base, so build a fresh "
                "pipeline with from_pretrained() first"
            )
        names = self._trainable_names(trainable_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        train_ids = self._record_ids(train_checked)
        import torch

        torch.manual_seed(seed)
        model, _ = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            keep = ("perplexity", "bits_per_token", "top1_accuracy", "top5_accuracy", "n_masked")
            return {
                k: v
                for k, v in self.evaluate(val_checked, mask_rate=mask_rate, seed=seed).items()
                if k in keep
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_ppl = entry["val"]["perplexity"] if entry["val"] else math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                masked = self._masked(train_checked, train_ids, rate=mask_rate, seed=seed + 1_000_003 * epoch)
                order = torch.randperm(len(masked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [masked[i] for i in order[start : start + batch_size]]
                    width = max(len(ids) for ids, _p, _t in batch)
                    input_ids = torch.full((len(batch), width), PAD_TOKEN_ID, dtype=torch.long)
                    attention = torch.zeros((len(batch), width), dtype=torch.long)
                    labels = torch.full((len(batch), width), -100, dtype=torch.long)
                    for row, (ids, positions, targets) in enumerate(batch):
                        input_ids[row, : len(ids)] = torch.tensor(ids)
                        attention[row, : len(ids)] = 1
                        labels[row, positions] = torch.tensor(targets)
                    out = model(
                        input_ids=input_ids.to(device),
                        attention_mask=attention.to(device),
                        labels=labels.to(device),
                    )
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["perplexity"] if entry["val"] else -math.inf
                if current < best_ppl or not entry["val"]:
                    best_ppl = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "objective": "masked language modelling (continued pre-training)",
            "trainable_layers": trainable_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "lowest validation masked perplexity"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "mask_rate": mask_rate,
            "n_train": len(train_checked),
            "n_train_tokens": sum(len(ids) - 2 for ids in train_ids),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted encoder-layer tensors as safetensors with a manifest naming the base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported
        format and version, the pinned base (id, revision, weight file, digest), exactly one file entry
        named `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not
        authenticity against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int):
            raise ValueError("artifact manifest does not record an integer trainable_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("bert.encoder.layer."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable encoder-layer tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BERTMaskedLMPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/bert_masked_lm_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Text-corpus dataset contract for continued masked-language-model training: the pinned SciTLDR sample of
paper abstracts, validation, seeded splitting, BYOD loaders and CSV export.

The default corpus is **real** and away from BERT's BooksCorpus + Wikipedia pre-training distribution: the
abstracts of computer-science papers shipped by SciTLDR (Cachola et al., EMNLP Findings 2020; Apache-2.0). The
three `SciTLDR-A` JSON-Lines files are fetched one by one from the project repository at a pinned commit and
refused on any byte-size or SHA-256 mismatch; only the abstract text is used here. The frozen model's masked
perplexity on held-out abstracts is the number to beat, and the fine-tuning question is whether a bounded
continued-MLM adaptation of the last encoder layers lowers it on abstracts the model has not seen.

A record is ``{id, text}``: one document of the domain, no label and no reference — masked-token prediction
needs none.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SciTLDR-A (paper abstracts)"
CORPUS_RELEASE = "allenai/scitldr @ 5ccad9c00a60ad75c9e04abf7f27d0f53f983b20"
CORPUS_BASE_URL = "https://raw.githubusercontent.com/allenai/scitldr/5ccad9c00a60ad75c9e04abf7f27d0f53f983b20/SciTLDR-Data/SciTLDR-A/"
CORPUS_FILES = {
    "train": ("train.jsonl", 3_155_015, "b222771d387be585cfdf5ae957b36757138415a352e0a3e3b23f73f87c3b1119"),
    "dev": ("dev.jsonl", 1_124_865, "3191fa98ccc09521332b7a1cd63b1930be4e8df125a235ccd31e40329709525e"),
    "test": ("test.jsonl", 1_204_107, "fb42dd6cd4f4a1928ae8a01a189456fbfe994a07e938bd49f68653933f6503c9"),
}
CORPUS_LICENSE = "Apache-2.0 (Cachola et al. 2020; allenai/scitldr)"
CORPUS_PAPERS = {"train": 1_992, "dev": 619, "test": 618}
DEFAULT_CACHE_DIR = Path("weights") / "scitldr"
MAX_SAMPLE_TEXT_CHARS = (
    2_000  # longer abstracts are left out of the sample (the ceiling is 512 WordPiece tokens)
)
MIN_SAMPLE_TEXT_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 300, "validation": 50, "test": 100}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
# BYOD: the notebook embeds two test documents as a pair and selects epochs on validation, so a split corpus
# needs at least this many test and validation records besides MIN_RECORDS training records.
MIN_TEST_RECORDS = 2
MIN_VAL_RECORDS = 1
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the three pinned SciTLDR-A files (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the JSON-Lines members into abstract records keeping each SciTLDR `paper_id`."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        records = []
        for line in files[split].decode("utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            records.append(
                {
                    "id": f"{split}-{row['paper_id']}",
                    "text": " ".join(str(s).strip() for s in row["source"]),
                    "paper_id": str(row["paper_id"]),
                }
            )
        if len(records) != CORPUS_PAPERS[split]:
            raise ValueError(f"{split}: {len(records)} papers, expected {CORPUS_PAPERS[split]}")
        out[split] = records
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Keep records whose text is within the sample length window; drop repeated texts case-insensitively."""
    seen: set[str] = set()
    kept = []
    for record in records:
        text = str(record["text"])
        if not MIN_SAMPLE_TEXT_CHARS <= len(text) <= MAX_SAMPLE_TEXT_CHARS:
            continue
        key = text.lower()
        if key in seen:
            continue
        seen.add(key)
        kept.append(dict(record))
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draws from the three SciTLDR members: training from `train`, validation from `dev`, test from
    `test` — the release's own paper-disjoint partition, re-checked on texts by `check_split_disjoint`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    source_of = {"train": "train", "validation": "dev", "test": "test"}
    rng = random.Random(seed)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        pool = filter_records(corpus[source_of[name]])
        if size > len(pool):
            raise ValueError(f"requested {size} {name} records but only {len(pool)} fit")
        rng.shuffle(pool)
        out[name] = [
            {"id": f"{name}-{i:04d}", "text": r["text"], "paper_id": r["paper_id"]}
            for i, r in enumerate(pool[:size])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/text")
    for key in ("id", "text"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, text = record["id"], record["text"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    if not isinstance(text, str):
        raise ValueError(f"{label}: text must be a string")
    if not text.strip():
        raise ValueError(f"{label}: text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{label}: text has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    item = {"id": rid, "text": text.strip()}
    if "paper_id" in record:
        item["paper_id"] = str(record["paper_id"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text corpus; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    texts: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        texts.add(item["text"].lower())
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_texts": len(texts),
        "text_chars": {
            "min": min(len(r["text"]) for r in checked),
            "max": max(len(r["text"]) for r in checked),
        },
        "total_chars": sum(len(r["text"]) for r in checked),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["text"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased text appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["text"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"a text ({record['text'][:60]!r}…) appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD corpus into train/validation/test after de-duplicating texts.

    The split must leave at least MIN_RECORDS training, MIN_VAL_RECORDS validation and MIN_TEST_RECORDS test
    records; a refusal names the split, the distinct-text count and the real minimum
    (`byod_minimum_records`)."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, min_records=1)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["text"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    needed = {"train": MIN_RECORDS, "validation": MIN_VAL_RECORDS, "test": MIN_TEST_RECORDS}
    short = {name: len(splits[name]) for name, least in needed.items() if len(splits[name]) < least}
    if short:
        name, have = next(iter(short.items()))
        try:
            minimum = byod_minimum_records(val_fraction, test_fraction)
            least = f"a corpus needs at least {minimum} distinct texts"
        except ValueError:
            least = "no corpus size gives every split its minimum at these fractions"
        raise ValueError(
            f"the {name} split has {have} records (at least {needed[name]} are required): {len(unique)} "
            f"distinct texts after de-duplication split into train/validation/test as "
            f"{len(splits['train'])}/{len(splits['validation'])}/{len(splits['test'])}; {least}. "
            "Add documents"
        )
    return splits


def byod_minimum_records(val_fraction: float = 0.15, test_fraction: float = 0.2) -> int:
    """Smallest number of distinct texts that `split_dataset` accepts at these fractions."""
    for n in range(1, MAX_RECORDS + 1):
        n_test = max(1, round(n * test_fraction))
        n_val = round(n * val_fraction)
        if n_test >= MIN_TEST_RECORDS and n_val >= MIN_VAL_RECORDS and n - n_test - n_val >= MIN_RECORDS:
            return n
    raise ValueError("no corpus within MAX_RECORDS satisfies the split")


def check_byod_tokens(
    splits: Mapping[str, Sequence[Mapping[str, Any]]], count_tokens: Any, *, max_tokens: int
) -> dict[str, Any]:
    """WordPiece budget of a split corpus, checked before any model call.

    `count_tokens(text)` returns the token count including [CLS]/[SEP]. A record over `max_tokens`, or with
    no token between [CLS] and [SEP], is refused naming its split and id: scoring and training reject such a
    record rather than truncate it."""
    over: list[str] = []
    empty: list[str] = []
    longest = 0
    for name, part in splits.items():
        for record in part:
            n = int(count_tokens(record["text"]))
            longest = max(longest, n)
            if n > max_tokens:
                over.append(f"{name}:{record['id']} ({n} tokens)")
            elif n < 3:
                empty.append(f"{name}:{record['id']}")
    if over:
        raise ValueError(
            f"{len(over)} record(s) exceed the {max_tokens}-token ceiling, which scoring and training reject "
            f"rather than truncate: {over[:10]}; split each long document into shorter ones (for example by "
            f"paragraph) or remove it"
        )
    if empty:
        raise ValueError(f"record(s) with no token between [CLS] and [SEP]: {empty[:10]}; remove them")
    return {
        "longest_tokens": longest,
        "ceiling": max_tokens,
        "records_checked": sum(len(part) for part in splits.values()),
    }


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, text}` records from a CSV (columns id, text), a JSON array or JSONL of such objects, or a
    plain `.txt` file in which every non-empty line (or blank-line-separated paragraph) is one record."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "text"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "text": r["text"]} for r in rows]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    if suffix == ".txt":
        paragraphs = [p.strip() for p in text.replace("\r\n", "\n").split("\n\n") if p.strip()]
        return [{"id": f"doc-{i:05d}", "text": " ".join(p.split())} for i, p in enumerate(paragraphs)]
    raise ValueError("BYOD corpora must be .csv, .json, .jsonl or .txt")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "text"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "text": record["text"]})
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `86b5e0934494…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bert-base-uncased",
  "modelId": "google-bert/bert-base-uncased",
  "revision": "86b5e0934494bd15c9632b12f734a8a67f723594",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 11356,
      "sha256": "43070e2d4e532684de521b885f385d0841030efa2b1a20bafb76133a5e1379c1"
    },
    {
      "path": "README.md",
      "bytes": 10517,
      "sha256": "9187b6018ea0010d884e78e098e328faa1b88b301570d0cce606bb35e4067e17"
    },
    {
      "path": "config.json",
      "bytes": 570,
      "sha256": "7160e1553ad2ca51d8c1cb066be533db31826e12d173824c1bb0cb1a4f187d20"
    },
    {
      "path": "coreml/fill-mask/float32_model.mlpackage/Manifest.json",
      "bytes": 617,
      "sha256": "4e9566ed44c3c401dada9ff243e0efbd570f9bc4e55dd7fdfb937cb23993c94c"
    },
    {
      "path": "model.safetensors",
      "bytes": 440449768,
      "sha256": "68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 466062,
      "sha256": "ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 48,
      "sha256": "a025160ef0431f1a392f6f050c1310f4c5d9fb6f275932dbccba73c4d214bf10"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 441170446
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'google-bert/bert-base-uncased', 'revision': '86b5e0934494bd15c9632b12f734a8a67f723594', 'license': 'apache-2.0', 'files': 8, 'total_bytes': 441170446}


{'weights_dir': '/content/weights/bert-base-uncased', 'fetched': ['LICENSE', 'README.md', 'config.json', 'coreml/fill-mask/float32_model.mlpackage/Manifest.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']}


{'verified_files': 8, 'revision': '86b5e0934494bd15c9632b12f734a8a67f723594'}


Some weights of the model checkpoint at /content/weights/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the three pinned SciTLDR-A files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` flattens each JSON-Lines member into records whose `text` is the abstract's sentences joined by a space — titles and TLDRs are left unread. `build_sample_dataset` keeps abstracts of 200..2,000 characters, drops repeated texts, and draws 300 training documents from the `train` member, 50 validation documents from `dev` and 100 test documents from `test` by a seeded shuffle — the release's own paper-disjoint partition. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no text appears in two splits, and the training split is written to `outputs/bert_masked_lm_train.csv` in the shape BYOD expects.

The cell also checks two things the model would otherwise only discover later, using the tokenizer and plain text only (no model call): every record's WordPiece count against the 512-token ceiling (a longer record is refused by split and id), and the **cloze** records Section 5 masks — the opening sentence of a record with its longest word of four or more letters replaced by `[MASK]`. With the sample these are three dev abstracts outside every split (*unseen*); a record whose opening sentence has no such word is skipped.

**Bring your own data:** set `USE_BYOD = True`, either leave `BYOD_PATH` empty to get an upload dialog (Colab only) or set it to a file already in this runtime, then select this cell and choose **Runtime → Run after**. `split_dataset` de-duplicates texts and makes a seeded 65 / 15 / 20 % split; it refuses a corpus that leaves fewer than 8 training, 2 validation or 2 test documents (at least 12 distinct documents), naming the split. Under BYOD there is no pool of unused documents, so the clozes come from your **test** records and are labelled that way (`byod-test-…`).

Look for: 1,992 + 619 + 618 raw papers, three digests, splits 300 / 50 / 100, the longest record's token count, three cloze records, and four refusal probes — a duplicate id, an empty text, a missing field and a dataset too small to split — each rejected before `torch` does anything.

**Predict before running:** the sample keeps abstracts of 200..2,000 characters. Will any of them come near the 512-token ceiling?

In [ ]:
import hashlib
import io
import json
import re

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
dropped_duplicates = 0
if USE_BYOD:
    if BYOD_PATH.strip():
        byod_path = Path(BYOD_PATH.strip()).expanduser()
        if not byod_path.is_file():
            raise FileNotFoundError(f'BYOD_PATH = {BYOD_PATH!r} is not a file in this runtime: fix the path, or leave BYOD_PATH empty to upload a file in Colab.')
        file_name = byod_path.name
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('The upload dialog needs Google Colab. Elsewhere, put the file in this runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError(f'Upload exactly one .csv, .json, .jsonl or .txt file (received {len(uploaded)}). Run this cell again and choose a file, or set BYOD_PATH to a file already in this runtime.')
        file_name, payload = next(iter(uploaded.items()))
        byod_path = Path('work') / Path(file_name).name
        byod_path.parent.mkdir(parents=True, exist_ok=True)
        byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    dropped_duplicates = len(records) - sum(len(part) for part in splits.values())
    data_source = 'BYOD (' + file_name + ')'
    raw_papers = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/scitldr'))
    raw_papers = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part, min_records=MIN_RECORDS if name == 'train' else 1) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
# Every record against the WordPiece ceiling, with the tokenizer only (no model call).
token_budget = check_byod_tokens(splits, pipe.count_tokens, max_tokens=MAX_TEXT_TOKENS)
write_dataset_csv(train_records, 'outputs/bert_masked_lm_train.csv')
print({'data_source': data_source, 'raw_papers': raw_papers, 'splits': disjoint, 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
print({'token_budget': token_budget, 'duplicate_texts_dropped': dropped_duplicates})

def cloze_of(record):
    """The opening sentence with its longest alphabetic word (4+ letters) replaced by [MASK]: (cloze, gold word), or None if it has no such word."""
    sentence = record['text'].split('. ')[0]
    words = [w for w in re.findall(r'[A-Za-z]+', sentence) if len(w) >= 4]
    if not words:
        return None
    gold = max(words, key=len)
    cloze = re.sub(r'\b' + gold + r'\b', MASK_TOKEN, sentence, count=1)
    return (cloze, gold.lower()) if cloze.count(MASK_TOKEN) == 1 else None

if USE_BYOD:
    cloze_pool, cloze_prefix = test_records[2:] + test_records[:2], 'byod-test'
    cloze_source = 'BYOD test records (scored in Sections 6 and 8; not unseen documents)'
else:
    used = {r['text'].lower() for part in splits.values() for r in part}
    cloze_pool, cloze_prefix = [r for r in filter_records(corpus['dev']) if r['text'].lower() not in used], 'unseen'
    cloze_source = 'unseen SciTLDR dev abstracts (in none of the splits)'
chosen, skipped_clozes = [], 0
for record in cloze_pool:
    made = cloze_of(record)
    if made is None:
        skipped_clozes += 1
        continue
    chosen.append((record, made))
    if len(chosen) == 3:
        break
if not chosen:
    raise ValueError('No test record opens with a sentence that has a word of four or more letters, so Section 5 has no cloze to fill. Start at least one document with an ordinary sentence ending in ". ", then run this cell again.')
unseen_records = [{**r, 'id': f'{cloze_prefix}-{i:02d}'} for i, (r, _made) in enumerate(chosen)]
clozes = {r['id']: made for r, (_record, made) in zip(unseen_records, chosen, strict=True)}
print({'clozes': len(clozes), 'cloze_source': cloze_source, 'records_skipped_without_a_usable_word': skipped_clozes})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_texts': manifest['unique_texts'], 'text_chars': manifest['text_chars'], 'total_chars': manifest['total_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {'id': train_records[0]['id'], 'text': train_records[0]['text'][:200] + '...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty text': [{**train_records[0], 'text': '   '}, *train_records[1:8]],
    'missing field': [{'id': r['id']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': 'SciTLDR-A (paper abstracts) (allenai/scitldr @ 5ccad9c00a60ad75c9e04abf7f27d0f53f983b20; Apache-2.0 (Cachola et al. 2020; allenai/scitldr))', 'raw_papers': {'train': 1992, 'dev': 619, 'test': 618}, 'splits': {'train': 300, 'validation': 50, 'test': 100}, 'file_sha256': {'train': 'b222771d387b...', 'dev': '3191fa98ccc0...', 'test': 'fb42dd6cd4f4...'}}
{'token_budget': {'longest_tokens': 411, 'ceiling': 512, 'records_checked': 450}, 'duplicate_texts_dropped': 0}
{'clozes': 3, 'cloze_source': 'unseen SciTLDR dev abstracts (in none of the splits)', 'records_skipped_without_a_usable_word': 0}
{'train': {'n': 300, 'unique_texts': 300, 'text_chars': {'min': 369, 'max': 1998}, 'total_chars': 327701, 'digest': '50798982fd48eea1...'}}
{'validation': {'n': 50, 'unique_texts': 50, 'text_chars': {'min': 348, 'max': 1788}, 'total_chars': 54420, 'digest': 'aea99f0e8be88055...'}}
{'test': {'n': 100, 'unique_texts': 100, 'text_chars': {'min': 341, 'max': 1832}, 'total_chars': 103710, '

**What to notice:** the longest record's token count against the 512-token ceiling, and that the splits share no text.

<details><summary>Check your reasoning</summary>No: in a local Windows CPU check of this notebook version the longest of the 450 sample records had 411 WordPiece tokens, so the ceiling is never reached (2,000 characters of English prose is roughly 400–450 tokens). A BYOD document of a few thousand words would exceed it and is refused here, by split and id, instead of failing in Section 6.</details>

## 5. Fill a mask and embed through the inference contracts

Before any adaptation, both inference contracts are exercised as they always were. The first **cloze** chosen in Section 4 is filled — the removed word is the *gold* word, which may be several WordPiece tokens, in which case no single candidate can match it (reported, never asserted). With the sample the clozes come from unseen abstracts; under BYOD they are your test records, and the ids say so. `validate_inputs` applies exactly the checks `fill_mask` and `embed` apply (text type and character ceiling, exactly one `[MASK]` for the cloze, `top_k` and `pooling` within their contracts) and returns an input manifest; the WordPiece ceiling `MAX_TEXT_TOKENS` (512) needs the real tokenizer and is enforced inside the pipeline, which **rejects, never truncates**. A two-mask input is validated too and its rejection recorded as a finding. `fill_mask` returns `top_k` candidates with a softmax `score` (a ranking signal, not a calibrated probability); `embed` returns one unit-norm 768-d vector per text under the requested pooling. The clozes and the embeddings of two test abstracts are kept as the *before* column for Section 9.

**Every pass starts from the pretrained model.** Section 7 changes the model in memory. If this cell, Section 6 or Section 7 runs again after Section 7 (a BYOD run or one of the optional experiments), it first reloads the pretrained model from the verified snapshot with `reset_to_pretrained()`, so a *before* or *frozen* number is never read from the adapted model.

**Predict before running:** for the masked word in the printed cloze, will the gold word be among the five candidates? Will the two abstracts' cosine be close to 0 or close to 1?

In [ ]:
import gc
import time

TOP_K = 5  # @param {type:"integer"}
POOLING = 'mean'  # @param ["cls", "mean"]

def reset_to_pretrained():
    """Sections 5-7 start from the pretrained model: if an earlier pass adapted `pipe`, reload it from the verified snapshot."""
    global pipe
    if pipe.adapter is None:
        return
    pipe = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pipe = BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
    print({'reloaded': 'the pretrained model, from the verified snapshot', 'reason': 'an earlier pass had adapted the model in memory'})

reset_to_pretrained()
cloze, gold = clozes[unseen_records[0]['id']]
pair_texts = [test_records[0]['text'], test_records[1]['text']]
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_BATCH': MAX_BATCH, 'MAX_TOP_K': MAX_TOP_K, 'VOCAB_SIZE': VOCAB_SIZE, 'HIDDEN_SIZE': HIDDEN_SIZE, 'MASK_TOKEN': MASK_TOKEN, 'MASK_TOKEN_ID': MASK_TOKEN_ID, 'POOLINGS': POOLINGS, 'DEFAULT_TOP_K': DEFAULT_TOP_K}
print(ceilings)
input_manifest = validate_inputs([cloze, *pair_texts], top_k=TOP_K, pooling=POOLING, names=['cloze', 'pair-a', 'pair-b'])
try:
    validate_inputs([f'A {MASK_TOKEN} and another {MASK_TOKEN}.'], top_k=TOP_K, pooling=POOLING)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'two-mask-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bert_masked_lm_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
filled = pipe.fill_mask(cloze, top_k=TOP_K)
fill_seconds = round(time.perf_counter() - started, 3)
started = time.perf_counter()
embedding_result = pipe.embed(pair_texts, pooling=POOLING)
embed_seconds = round(time.perf_counter() - started, 3)
vectors = np.asarray(embedding_result['embeddings'], dtype=np.float32)
scores = [c['score'] for c in filled['candidates']]
checks = {
    'top_k_candidates_returned': len(filled['candidates']) == TOP_K,
    'scores_descending_in_unit_interval': all(a >= b for a, b in zip(scores, scores[1:], strict=False)) and all(0.0 < s <= 1.0 for s in scores),
    'tokens_within_vocab': all(0 <= c['token_id'] < VOCAB_SIZE for c in filled['candidates']),
    'n_tokens_within_ceiling': 1 <= filled['n_tokens'] <= MAX_TEXT_TOKENS and all(1 <= n <= MAX_TEXT_TOKENS for n in embedding_result['n_tokens']),
    'one_unit_vector_per_text': vectors.shape == (2, HIDDEN_SIZE) and bool(np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4)),
    'pooling_as_requested': embedding_result['pooling'] == POOLING and embedding_result['dim'] == HIDDEN_SIZE,
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
print({'cloze': cloze, 'gold_word': gold, 'candidates': [(c['token'], round(c['score'], 4)) for c in filled['candidates']], 'gold_in_top_k': gold in [c['token'] for c in filled['candidates']], 'seconds': fill_seconds})
print({'pair_cosine': round(float(vectors[0] @ vectors[1]), 4), 'pooling': POOLING, 'n_tokens': embedding_result['n_tokens'], 'seconds': embed_seconds, 'checks': checks, 'findings': len(input_manifest['findings']), 'score_semantics': 'softmax ranking signal, not a calibrated probability; no threshold shipped'})
before = {rid: [c['token'] for c in pipe.fill_mask(text, top_k=TOP_K)['candidates']] for rid, (text, _gold) in clozes.items()}
vectors_before = vectors
print({'clozes_filled_by_the_frozen_model': len(before), 'cloze_source': cloze_source})

{'MAX_TEXT_CHARS': 4000, 'MAX_TEXT_TOKENS': 512, 'MAX_BATCH': 64, 'MAX_TOP_K': 100, 'VOCAB_SIZE': 30522, 'HIDDEN_SIZE': 768, 'MASK_TOKEN': '[MASK]', 'MASK_TOKEN_ID': 103, 'POOLINGS': ('cls', 'mean'), 'DEFAULT_TOP_K': 5}


{'cloze': 'Many real-world [MASK], e.g', 'gold_word': 'problems', 'candidates': [('systems', 0.107), ('situations', 0.0599), ('phenomena', 0.0572), ('events', 0.0568), ('objects', 0.0566)], 'gold_in_top_k': False, 'seconds': 0.279}
{'pair_cosine': 0.8624, 'pooling': 'mean', 'n_tokens': [175, 337], 'seconds': 0.085, 'checks': {'top_k_candidates_returned': True, 'scores_descending_in_unit_interval': True, 'tokens_within_vocab': True, 'n_tokens_within_ceiling': True, 'one_unit_vector_per_text': True, 'pooling_as_requested': True}, 'findings': 1, 'score_semantics': 'softmax ranking signal, not a calibrated probability; no threshold shipped'}
{'clozes_filled_by_the_frozen_model': 3, 'cloze_source': 'unseen SciTLDR dev abstracts (in none of the splits)'}


**What to notice:** the five candidates are all plausible words for the slot even when the gold word is missing, and the scores fall quickly after the first.

<details><summary>Check your reasoning</summary>In the recorded CPU pre-flight of the previous notebook version and in a local Windows CPU check of this one, the first cloze's gold word was `problems` and the pretrained top five were `systems situations phenomena events objects`: grammatical, on-topic, and wrong. The score is a softmax over 30,522 tokens, a ranking signal rather than a probability of being right: the top candidate scored only 0.107 there, and even a high score would not mean that chance of being correct. Mean-pooled embeddings of two abstracts from one field are usually close to 1 (0.86 for this pair in that run): cosine between BERT vectors is high for almost any two English texts, which is one reason the notebook calls embeddings representations, not judgements of similarity.</details>

## 6. The unigram floor and the frozen model's masked-token metrics on the test split

Two numbers frame the adaptation. `pipe.unigram_baseline` fits an add-one-smoothed unigram model over the 30,522-token vocabulary on the training tokens and predicts every masked position of the test abstracts with it: the perplexity and top-k accuracy a model that knows the domain's word frequencies but ignores every context reaches — expect a perplexity in the thousands and a top-1 accuracy of a few percent (the commonest tokens). `pipe.evaluate` masks a seeded 15 % of each test abstract's interior WordPiece tokens (`record_seed` makes the positions a function of the record id and `SEED`, so every model scored here sees the same masks), runs the frozen encoder and masked-LM head once per record, and reads the negative log-likelihood and the rank of the original token at every masked position: **masked perplexity** (`exp` of the mean NLL), **bits per masked token** and **top-1 / top-5 accuracy**. Records over `MAX_TEXT_TOKENS` are refused, never truncated (Section 4 already checked). The cell reloads the pretrained model first if an earlier pass adapted it, and refuses to report an adapted model as frozen. Scoring the 100 test abstracts took 6.6 s in the local CPU pre-flight of the previous notebook version.

**Predict before running:** roughly what masked perplexity will a model that ignores context entirely (the unigram floor) reach — tens, hundreds or thousands? And the pretrained model?

In [ ]:
SEED = 0  # @param {type:"integer"}
MASK_RATE = 0.15  # @param {type:"number"}

def brief(m):
    return {'perplexity': round(m['perplexity'], 2), 'bits_per_token': round(m['bits_per_token'], 3), 'top1_accuracy': round(m['top1_accuracy'], 4), 'top5_accuracy': round(m['top5_accuracy'], 4), 'n_masked': m['n_masked']}

reset_to_pretrained()
t0 = time.perf_counter()
unigram = pipe.unigram_baseline(train_records, test_records, mask_rate=MASK_RATE, seed=SEED)
print({'unigram_floor': brief(unigram), 'baseline': unigram['baseline'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, mask_rate=MASK_RATE, seed=SEED)
if frozen_test['adapted']:
    raise RuntimeError('Section 6 must score the pretrained model, but the model in memory is adapted. Restart the session and choose Run all.')
if frozen_test['n_masked'] != unigram['n_masked']:
    raise RuntimeError(f"The floor and the frozen model were scored at different masked positions ({unigram['n_masked']} vs {frozen_test['n_masked']}); re-run this cell without changing SEED or MASK_RATE in between.")
print({'frozen_model_test': brief(frozen_test), 'n_records': frozen_test['n_records'], 'verdict': frozen_test['verdict'], 'adapted': frozen_test['adapted'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'frozen_below_unigram_floor': frozen_test['perplexity'] < unigram['perplexity']})
print({'definitions': frozen_test['definitions']})

{'unigram_floor': {'perplexity': 1174.57, 'bits_per_token': 10.198, 'top1_accuracy': 0.0422, 'top5_accuracy': 0.1488, 'n_masked': 2983}, 'baseline': 'add-one-smoothed unigram model fitted on the training tokens (ignores context)', 'seconds': 0.6}


{'frozen_model_test': {'perplexity': 15.09, 'bits_per_token': 3.916, 'top1_accuracy': 0.5334, 'top5_accuracy': 0.7013, 'n_masked': 2983}, 'n_records': 100, 'verdict': 'measured', 'adapted': False, 'seconds': 1.9}
{'frozen_below_unigram_floor': True}
{'definitions': {'perplexity': 'exp of the mean negative log-likelihood (natural log) of the original token at every masked position of the held-out records, all masked positions weighted equally; lower is better', 'bits_per_token': 'the same mean negative log-likelihood divided by ln 2', 'mean_nll': 'the mean negative log-likelihood at the masked positions in nats', 'top1_accuracy': "fraction of masked positions whose original token is the model's first candidate", 'top5_accuracy': 'fraction of masked positions whose original token is among the first five candidates', 'masking': "a seeded 15 % of each record's WordPiece tokens (never [CLS]/[SEP], at least one token) replaced by [MASK] — the same positions for every model scored under the s

**What to notice:** the gap between the floor and the pretrained model, and `adapted: False`.

<details><summary>Check your reasoning</summary>Thousands for the floor: in the recorded runs of the previous notebook version (same model, data and seed) the unigram floor scored a masked perplexity of about 1,175 with top-1 accuracy 4.2 %, and the pretrained model about 15.09 with top-1 53.3 % (Kaggle T4 and the CPU pre-flight agree). A perplexity of 1,175 means the floor is, on average, as unsure as a uniform choice among about 1,175 tokens: knowing only how frequent words are, it can guess `the` and `of` but nothing that depends on the sentence. The pretrained encoder reads the words on both sides of each gap, which narrows the choice to about 15 equally likely tokens — BooksCorpus and Wikipedia already contain a lot of technical prose.</details>

## 7. Bounded continued masked-language-model training

`pipe.adapt` trains only the last `TRAINABLE_LAYERS` encoder layers — four by default, 28,351,488 of 109,514,298 parameters; the word, position and token-type embeddings, the earlier layers and the masked-LM head with its decoder tied to the word embeddings stay frozen — with the model's own pre-training objective: every epoch re-draws a seeded `MASK_RATE` of each training abstract's interior tokens, replaces them with `[MASK]` and applies cross-entropy at those positions only (100 % `[MASK]`, not the upstream 80/10/10 replacement mix), AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Epoch 0 records the frozen model's validation metrics under the evaluation masking; every epoch is scored the same way, and the epoch with the lowest validation masked perplexity is kept.

Watch validation perplexity fall from about 13 by a point or two over two epochs while top-1 accuracy creeps up (about 100 s of training plus a validation pass per epoch in the local CPU pre-flight of the previous notebook version; much less on a GPU). That pre-flight's sweep on this sample (adapted test perplexity, local Windows CPU): two layers at 1e-4 reached 12.96, four layers at 1e-4 reached 12.51 and four layers at 5e-5 reached 12.59 in about the same time — the default keeps the standard 5e-5.

**Adaptation always starts from the pretrained model.** If this cell runs again (for example with `TRAINABLE_LAYERS = 1` in Section 10), it first reloads the pretrained model from the verified snapshot, so the new training never stacks on the previous one, epoch 0 really is the frozen model, and the exported adapter holds exactly the layers this run trained; `pipe.adapt` itself refuses a model that is already adapted.

**Predict before running:** will validation perplexity fall in both epochs? Which epoch will be kept?

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 4  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_perplexity'] = round(entry['val']['perplexity'], 2)
        row['val_top1'] = round(entry['val']['top1_accuracy'], 4)
        row['val_top5'] = round(entry['val']['top5_accuracy'], 4)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

reset_to_pretrained()
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_layers=TRAINABLE_LAYERS, mask_rate=MASK_RATE, seed=SEED, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'objective': adapt_result['objective'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'train_tokens': adapt_result['n_train_tokens'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

{'epoch': 0, 'train_loss': None, 'val_perplexity': 12.89, 'val_top1': 0.545, 'val_top5': 0.7176, 'note': 'frozen model'}


{'epoch': 1, 'train_loss': 2.6178, 'val_perplexity': 11.14, 'val_top1': 0.5574, 'val_top5': 0.7232}


{'epoch': 2, 'train_loss': 2.6072, 'val_perplexity': 10.91, 'val_top1': 0.5611, 'val_top5': 0.7337}
{'objective': 'masked language modelling (continued pre-training)', 'trainable_parameters': 28351488, 'total_parameters': 109514298, 'train_tokens': 63625, 'best_epoch': 2, 'selection': 'lowest validation masked perplexity', 'seconds': 25.5}


**What to notice:** epoch 0 is labelled the frozen model, and the kept epoch is the one with the lowest validation perplexity — the test split plays no part in that choice.

<details><summary>Check your reasoning</summary>In the local CPU pre-flight of the previous notebook version validation perplexity went 12.89 → 11.15 → 10.87 over epochs 0, 1 and 2, so it fell in both epochs and epoch 2 was kept. It need not: with more epochs or a higher learning rate it can rise again, and the rule then keeps an earlier epoch (epoch 0 means no training change is kept). Choosing the epoch on the test split would make the Section 8 comparison optimistic, because the test abstracts would have helped pick the model they then judge.</details>

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no abstract in it appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6 — same masks, same seed — and the three rows are put side by side. Look for a masked perplexity a point or two below the frozen one with top-1 and top-5 accuracy up by about a point, all far from the unigram floor. The cell prints a verdict — *improved* or *not improved* — and carries on either way: a flat or negative result from an experiment is a result to read, not an error. It also keeps a **run history** for this session, one row per pass through Section 8, so an experiment can be compared with the default run (Section 10). One hundred abstracts from one seeded split of one corpus give no dispersion estimate; the delta is sample-sanity evidence that the adaptation contract works, not a benchmark, and a lower masked perplexity on paper abstracts says nothing about your corpus until you measure it there.

**Predict before running:** how many points of masked perplexity will four adapted layers gain on the test abstracts — less than 1, about 2–3, or more than 5?

In [ ]:
adapted_test = pipe.evaluate(test_records, mask_rate=MASK_RATE, seed=SEED)
adapted_val = pipe.evaluate(val_records, mask_rate=MASK_RATE, seed=SEED)
comparison = {
    metric: {'unigram_floor': round(unigram[metric], 4), 'frozen': round(frozen_test[metric], 4), 'adapted': round(adapted_test[metric], 4)}
    for metric in ('perplexity', 'bits_per_token', 'top1_accuracy', 'top5_accuracy')
}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('perplexity', 'bits_per_token', 'top1_accuracy', 'top5_accuracy')}
for metric, row in comparison.items():
    print({metric: row})
improved = adapted_test['perplexity'] < frozen_test['perplexity']
verdict = 'improved: the adapted masked perplexity is below the frozen one' if improved else 'not improved: the adapted masked perplexity is not below the frozen one (read it; the notebook carries on)'
print({'verdict': verdict})
run_history = globals().get('run_history', [])
run_history.append({'run': len(run_history) + 1, 'data': 'BYOD' if USE_BYOD else 'sample', 'trainable_layers': TRAINABLE_LAYERS, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'seed': SEED, 'trainable_parameters': adapt_result['n_trainable'], 'best_epoch': adapt_result['best_epoch'], 'frozen_perplexity': round(frozen_test['perplexity'], 3), 'adapted_perplexity': round(adapted_test['perplexity'], 3), 'delta': round(adapted_test['perplexity'] - frozen_test['perplexity'], 3)})
print(f"{'run':>3} {'data':>6} {'layers':>6} {'epochs':>6} {'lr':>8} {'seed':>4} {'best':>4} {'frozen':>8} {'adapted':>8} {'delta':>7}")
for h in run_history:
    print(f"{h['run']:>3} {h['data']:>6} {h['trainable_layers']:>6} {h['epochs']:>6} {h['learning_rate']:>8g} {h['seed']:>4} {h['best_epoch']:>4} {h['frozen_perplexity']:>8.3f} {h['adapted_perplexity']:>8.3f} {h['delta']:>7.3f}")
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'masking': {'mask_rate': MASK_RATE, 'seed': SEED, 'n_masked_test': frozen_test['n_masked']},
    'baselines': {'unigram_floor': unigram},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'verdict': verdict,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/bert_masked_lm_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'report': 'outputs/bert_masked_lm_evaluation_report.json'})

{'perplexity': {'unigram_floor': 1174.5675, 'frozen': 15.0932, 'adapted': 12.6567}}
{'bits_per_token': {'unigram_floor': 10.1979, 'frozen': 3.9158, 'adapted': 3.6618}}
{'top1_accuracy': {'unigram_floor': 0.0422, 'frozen': 0.5334, 'adapted': 0.5448}}
{'top5_accuracy': {'unigram_floor': 0.1488, 'frozen': 0.7013, 'adapted': 0.7211}}
{'delta_vs_frozen': {'perplexity': -2.4365, 'bits_per_token': -0.254, 'top1_accuracy': 0.0114, 'top5_accuracy': 0.0198}}
{'verdict': 'improved: the adapted masked perplexity is below the frozen one'}
run   data layers epochs       lr seed best   frozen  adapted   delta
  1 sample      4      2    5e-05    0    2   15.093   12.657  -2.437
{'report': 'outputs/bert_masked_lm_evaluation_report.json'}


**What to notice:** the three rows of the comparison, the verdict, and how small the accuracy gains are next to the perplexity drop.

<details><summary>Check your reasoning</summary>About 2–3 points. In the recorded runs of the previous notebook version the test masked perplexity fell from 15.09 to 12.59 in the local CPU pre-flight and to 12.66 on the Kaggle T4 (top-1 53.3 → 54.8 % and 53.3 → 54.5 %). The small CPU/GPU difference comes from non-deterministic GPU arithmetic during training, not from an error. Perplexity moves more than top-1 accuracy because it also rewards giving the right token more probability when it is not the first choice. One hundred abstracts with no dispersion estimate cannot separate two configurations a few tenths of a point apart.</details>

## 9. Clozes and embeddings before and after, export the adapter and reload it

The clozes filled by the frozen model in Section 5 are filled again by the adapted model through the same `fill_mask` contract, printed side by side with the gold word and whether it appears in the top-`k` before and after; the two test abstracts are re-embedded and their cosine similarity and each vector's cosine to its frozen self are printed. Read these as observations: the adapter moves the last encoder layers, so **every embedding changes** — how much, and whether for the better on your similarity task, nothing here measures. The single-input `evaluation_report` helper — the inference-stage helper — is written for the first cloze and stays `not-measurable`, because one cloze has no metric.

`pipe.save_artifact` writes the trained tensors — the last four encoder layers, about 113 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `BERTMaskedLMPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an encoder-layer tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell checks for an identical test perplexity on ten records, identical candidates and identical embeddings (VER4), and stops with an explanation if they differ. Under BYOD the clozes are your test records, already scored in Sections 6 and 8, and are labelled that way, not as unseen text.

**Predict before running:** will adaptation change the candidates of every cloze? Will each abstract's adapted embedding still be close to its frozen one?

In [ ]:
import csv
import shutil

rows = []
for record in unseen_records:
    text, gold_word = clozes[record['id']]
    after = [c['token'] for c in pipe.fill_mask(text, top_k=TOP_K)['candidates']]
    rows.append({'id': record['id'], 'cloze': text, 'gold_word': gold_word, 'frozen_candidates': ' '.join(before[record['id']]), 'adapted_candidates': ' '.join(after), 'gold_in_top_k_frozen': gold_word in before[record['id']], 'gold_in_top_k_adapted': gold_word in after})
    print({k: rows[-1][k] for k in ('id', 'gold_word', 'frozen_candidates', 'adapted_candidates', 'gold_in_top_k_frozen', 'gold_in_top_k_adapted')})
vectors_after = np.asarray(pipe.embed(pair_texts, pooling=POOLING)['embeddings'], dtype=np.float32)
embedding_shift = {'pair_cosine_frozen': round(float(vectors_before[0] @ vectors_before[1]), 4), 'pair_cosine_adapted': round(float(vectors_after[0] @ vectors_after[1]), 4), 'self_cosine_frozen_vs_adapted': [round(float(vectors_before[i] @ vectors_after[i]), 4) for i in range(2)]}
single_report = evaluation_report(pipe.fill_mask(cloze, top_k=TOP_K), [gold], sample_kind='one unseen SciTLDR abstract cloze' if not USE_BYOD else 'one BYOD test record (already scored in Sections 6 and 8; not unseen)')
print({'embedding_shift': embedding_shift, 'single_input_report_verdict': single_report['verdict'], 'clozes_with_changed_candidates': sum(r['frozen_candidates'] != r['adapted_candidates'] for r in rows), 'of': len(rows)})
with open('outputs/bert_masked_lm_cloze.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/bert_masked_lm_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'bert_masked_lm', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = BERTMaskedLMPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_records = test_records[:10]
ppl_pair = (pipe.evaluate(parity_records, mask_rate=MASK_RATE, seed=SEED)['perplexity'], reloaded.evaluate(parity_records, mask_rate=MASK_RATE, seed=SEED)['perplexity'])
cand_pair = [(r['adapted_candidates'], ' '.join(c['token'] for c in reloaded.fill_mask(r['cloze'], top_k=TOP_K)['candidates'])) for r in rows]
vectors_reloaded = np.asarray(reloaded.embed(pair_texts, pooling=POOLING)['embeddings'], dtype=np.float32)
parity = {'perplexity_in_memory': round(ppl_pair[0], 6), 'perplexity_reloaded': round(ppl_pair[1], 6), 'identical_candidates': sum(a == b for a, b in cand_pair), 'of': len(cand_pair), 'embeddings_identical': bool(np.array_equal(vectors_after, vectors_reloaded))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
if not (abs(ppl_pair[0] - ppl_pair[1]) < 1e-6 and parity['identical_candidates'] == parity['of'] and parity['embeddings_identical']):
    raise RuntimeError(f'Reload parity failed: {parity}. The saved adapter does not reproduce the model that was evaluated in Section 8. Re-run from Section 7 (it reloads the pretrained model before adapting), or restart the session and choose Run all.')

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'token_budget': token_budget,
    'duplicate_texts_dropped': dropped_duplicates,
    'cloze_source': cloze_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'cloze': cloze, 'gold_word': gold, 'fill_mask': {k: filled[k] for k in ('candidates', 'top_k', 'n_tokens')}, 'embed': {k: embedding_result[k] for k in ('dim', 'pooling', 'normalized', 'n_tokens')}, 'seconds': {'fill_mask': fill_seconds, 'embed': embed_seconds}},
    'comparison': comparison,
    'clozes_before_after': rows,
    'embedding_shift': embedding_shift,
    'single_input_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/bert_masked_lm_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

{'id': 'unseen-00', 'gold_word': 'problems', 'frozen_candidates': 'systems situations phenomena events objects', 'adapted_candidates': 'applications problems phenomena systems situations', 'gold_in_top_k_frozen': False, 'gold_in_top_k_adapted': True}
{'id': 'unseen-01', 'gold_word': 'foveation', 'frozen_candidates': 'it this vision photography depth', 'adapted_candidates': 'it this depth photography vision', 'gold_in_top_k_frozen': False, 'gold_in_top_k_adapted': False}
{'id': 'unseen-02', 'gold_word': 'normalization', 'frozen_candidates': '##ing ##nor ##on learning processing', 'adapted_candidates': '##nor ##ing ##on ##net ##am', 'gold_in_top_k_frozen': False, 'gold_in_top_k_adapted': False}
{'embedding_shift': {'pair_cosine_frozen': 0.8624, 'pair_cosine_adapted': 0.882, 'self_cosine_frozen_vs_adapted': [0.946, 0.9365]}, 'single_input_report_verdict': 'not-measurable', 'clozes_with_changed_candidates': 3, 'of': 3}


{'artifact': 'outputs/bert_masked_lm_adapter', 'format': 'org.valcorza.bert-base-uncased.adapter.v1', 'tensors': 64, 'bytes': 113413392, 'sha256': '442dd11c22678952...'}


Some weights of the model checkpoint at /content/weights/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'reload_parity': {'perplexity_in_memory': 14.735343, 'perplexity_reloaded': 14.735343, 'identical_candidates': 3, 'of': 3, 'embeddings_identical': True}, 'reloaded_best_epoch': 2}
['bert_masked_lm_adapter', 'bert_masked_lm_cloze.csv', 'bert_masked_lm_evaluation_report.json', 'bert_masked_lm_input_manifest.json', 'bert_masked_lm_result.json', 'bert_masked_lm_train.csv']


**What to notice:** which clozes changed their candidates, each vector's cosine to its frozen self, and the reload-parity line.

<details><summary>Check your reasoning</summary>Not necessarily every cloze: in the local CPU pre-flight of the previous notebook version `problems` entered the first cloze's top five after adaptation, while the multi-piece gold words `foveation` and `normalization` matched neither before nor after (no single candidate can). The embeddings do all move, because the adapted layers are shared by every input, but only a little: each abstract's adapted vector had a cosine of about 0.94 to its frozen self. Whether that move helps a similarity task is not measured here. Reload parity was exact (identical ten-record perplexity, candidates and embeddings): the exported adapter is the model that was evaluated.</details>

## 10. Your turn — change one thing: fewer trainable layers

Optional; **Predict → Change → Run → Observe → Explain**. It retrains, so it takes about as long as Section 7 did.

1. **Predict:** with only the last encoder layer trainable instead of four, will the test perplexity gain be about the same, about half, or close to zero? Write your guess down, next to the default run's numbers in the Section 8 run history.
2. **Change:** in Section 7 set `TRAINABLE_LAYERS = 1`. Change nothing else.
3. **Run:** select the Section 7 cell and choose **Runtime → Run after**. Section 7 reloads the pretrained model before adapting, Section 8 compares against the same Section 6 frozen score and adds a row to the run history, and Section 9 exports and reloads the new adapter.
4. **Observe:** in Section 7, epoch 0 shows the *frozen* validation perplexity again (about 12.9, not the adapted 10.9); in Section 8, the run history now has two rows — compare `trainable_parameters` and `delta`; in Section 9, the adapter holds 16 tensors instead of 64 and reload parity still holds.
5. **Explain:** in one sentence, why does training fewer layers change the gain the way it did?

<details><summary>Check your reasoning</summary>About two thirds of the gain: not all of it, and not zero. In a local Windows CPU check of this notebook version (float32, real weights, default settings otherwise) the default run gave 15.09 → 12.59 (delta −2.50) with four layers, and the one-layer re-run gave 15.09 → 13.36 (delta −1.73); its epoch 0 showed the frozen validation perplexity 12.89, validation perplexity went 12.89 → 11.77 → 11.45, the adapter held 16 tensors (28.4 MB instead of 113 MB) and reload parity was exact. One layer has a quarter of the trainable parameters (7,087,872 of 109,514,298) and can only re-weight the last layer's view of the text, so it adapts less to the new domain in the same number of steps; that the last layer alone still gives most of the gain suggests much of the domain shift is in how the final layer maps context to words. If your gain did not shrink, check that epoch 0 showed the frozen validation perplexity; if it did not, the model was not reset (restart and **Run all**).</details>

## Interpretation and limits

The frozen encoder already predicts the masked words of paper abstracts far better than a unigram model does (a masked perplexity near 15 against a floor in the thousands, top-1 accuracy above 50 % against a few percent — BooksCorpus and Wikipedia contain technical prose), and a bounded continuation of its own pre-training objective on 300 abstracts, touching only the last four encoder layers, lowers held-out masked perplexity by about 2.5 points (15.09 → 12.59 in the local CPU pre-flight of the previous notebook version, 15.09 → 12.66 on its Kaggle T4 run), with a 113 MB adapter that reloads to identical likelihoods, candidates and embeddings. That is the claim: the adaptation contract can continue pre-training end to end on a real corpus, and the number it produces is read against the frozen model and a context-free floor rather than in isolation.

Masked perplexity is intrinsic: it says how well the encoder predicts missing tokens of text it did not write, at one seeded 15 % masking of one seeded split of one corpus with no dispersion estimate. It is not embedding quality, retrieval performance or downstream accuracy, and a lower value does not make the candidates or the cosine table in Section 9 better — those are observations. The adapter changes the last layers, which every input shares, so every embedding shifts (Section 9 prints each vector's cosine to its frozen self) and the fill-mask `score` stays a softmax ranking signal, not a probability. Text is lower-cased and accent-stripped by the tokenizer; texts over 512 WordPiece tokens are rejected everywhere, never truncated; the checkpoint carries the gender and occupation associations the upstream card documents, and continued pre-training on a corpus does not remove them.

Three things to carry to real data. **Floors first:** the unigram floor and the frozen masked perplexity on *your* held-out documents are the numbers to read before any adapted one. **Leakage:** de-duplicate texts across splits (the contract does this case-insensitively) and split by document collection or author when your documents come from one. **Downstream:** a better masked perplexity on your domain is a reason to *measure* your similarity or classification task with the adapted encoder, not evidence that it improved.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real corpus, validate the demonstrated dataset contract without leakage, execute both inference contracts and a bounded continued pre-training, evaluate by masked-token metrics against a trivial floor and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, embedding or cloze quality on any domain, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path).** Each one changes one form field; note the numbers you want to compare first (the Section 8 run history keeps one row per pass). In Colab, **Runtime → Run after** runs the selected cell and every cell below it; Sections 5, 6 and 7 reload the pretrained model before they use it, so no experiment inherits an earlier adaptation.

- **Fewer trainable layers:** Section 10.
- **More epochs:** set `EPOCHS = 4` in Section 7, select Section 7 and choose **Run after**; watch whether validation perplexity keeps falling or turns (the best epoch is kept either way).
- **Other masks:** change `SEED` in Section 6, select Section 6 and choose **Run after**; read how much the frozen and adapted metrics move with a different set of masked positions.
- **Other pooling:** switch `POOLING` in Section 5, select Section 5 and choose **Run after**; re-read the pair cosine and the cosine shift in Section 9.
- **Your own documents:** BYOD (Section 4, then **Run after**); read the unigram floor and the frozen perplexity before the adapted number.

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment is built from manylinux wheels. Use Google Colab, Kaggle or a Linux Jupyter server; Windows and macOS kernels are not supported.
- **Section 1 fails while downloading** (`uv` wheel, Python build or packages). The runtime needs PyPI and the python-build-standalone download; run the cell again once the network is back. A "size/SHA-256" or "does not match its digest" error means a file was altered: do not edit the cell, regenerate the notebook from the repository.
- **"The isolated environment's Python process exited".** Usually the runtime ran out of memory. Restart the session and choose **Run all**; on CPU, close other notebooks first.
- **Out of disk, or the checkpoint download stops.** The locked install and the 440 MB checkpoint need a few GB of free space. Start a fresh runtime, or delete `dimer_isolated_env/` and `weights/` from an earlier attempt.
- **A digest or size mismatch in Section 3 or 4.** A download was incomplete or altered; the notebook refuses it rather than continuing. Run the cell again; never edit the manifest or the corpus digests.
- **BYOD is refused in Section 4.** The message names what to fix: a split left with too few documents (a corpus needs at least 12 distinct documents), a record over the 512-token ceiling (by split and id: split the document by paragraph), no test record with a usable opening sentence for the clozes, a missing column or a duplicate id. Fix the file and run Section 4 with **Run after** again. An empty or cancelled upload asks you to choose a file or set `BYOD_PATH`.
- **Section 8 says "not improved".** Not an error: the adapted model did not beat the frozen one under these settings. Read the epoch history in Section 7; the export and reload in Section 9 still run.
- **Reload parity fails in Section 9.** The saved adapter does not reproduce the evaluated model. Re-run from Section 7 (it reloads the pretrained model first), or restart the session and choose **Run all**.
- **Numbers differ slightly from the recorded runs.** GPU training is not bit-for-bit deterministic: a tenth of a perplexity point between a T4 and a CPU run is expected. Larger differences on the default settings mean a form field was changed.
- **You re-ran the routing cell in Section 1.** It starts a new isolated process and discards every variable created so far; choose **Run all** again.

## Glossary

- **`[MASK]` / cloze:** the special token that hides a word, and a sentence with one word hidden for the model to fill.
- **WordPiece:** BERT's tokenizer; it splits rare words into several pieces, so one word can be several tokens.
- **Softmax score:** the model's scores over all 30,522 tokens turned into numbers that sum to one; a ranking signal, not a calibrated probability of being right.
- **Masked perplexity:** the exponential of the average negative log-likelihood of the hidden tokens; lower is better, and a value of *k* means the model is on average as unsure as a uniform choice among *k* tokens.
- **Bits per masked token:** the same average expressed in bits.
- **Top-1 / top-5 accuracy:** the share of hidden tokens that are the model's first choice / among its first five.
- **Unigram floor:** a model that predicts every hidden token from word frequencies alone, ignoring the context; the score any useful model must beat.
- **Continued pre-training:** further training on the model's own masked-word objective, with a new domain's unlabelled text.
- **Trainable layers:** the encoder layers whose weights training may change; the rest stay frozen.
- **Epoch / validation selection:** one pass over the training documents; the epoch kept is the one with the lowest validation perplexity.
- **Training / validation / test split:** documents used to train, to choose the epoch, and to report the final result once.
- **Embedding / pooling / cosine:** a fixed-length vector for a text, how the per-token vectors are combined into it (`cls` or `mean`), and the similarity of two unit vectors (1 = same direction).
- **Adapter / reload parity:** the saved trained layers, and the check that a pipeline rebuilt from them gives the same perplexity, candidates and embeddings.

## Conclusion (your notes)

Optional personal notes, not a required submission. Complete each sentence from your own run:

- On these test abstracts the unigram floor reached a masked perplexity of ___, the pretrained model ___ and the adapted model ___; the verdict was ___.
- Adapting ___ layer(s) changed the perplexity by ___; with ___ layer(s) (Section 10) it changed by ___, because ___.
- A lower masked perplexity tells me ___, but it does not tell me ___.
- Before trusting this adaptation on my own documents I would measure ___ on ___.

## References

- Repository README: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google-bert/bert-base-uncased
- Upstream code: https://github.com/google-research/bert
- BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding (Devlin et al., NAACL 2019): https://arxiv.org/abs/1810.04805
- TLDR: Extreme Summarization of Scientific Documents (Cachola et al., EMNLP Findings 2020; SciTLDR, Apache-2.0): https://arxiv.org/abs/2004.15011
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)